# Phase 2-ZK: λ_d + ZeckendorfReadout

**Что делает**: берёт чекпоинт Phase2 (`model_step25000.pt`), заменяет lm_head (44.8M) на ZeckendorfReadout (86K), дообучает 3 эпохи.
**Цель**: сравнить две архитектуры head-to-head на одинаковых данных.

**Порядок ячеек:**
1. Монтировать Drive
2. Клонировать репозиторий
3. Скопировать данные с Drive
4. Проверить GPU
5. **Запустить тренировку Phase2-ZK** (~2-3 часа на T4)
6. Сравнить lm_head vs Zeckendorf
7. Сохранить результаты на Drive

In [ ]:
# === 1. Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

import os
# Auto-find data directory
for candidate in ['lambda', 'eva_lambda', 'EVA-Ai', 'data']:
    p = f'/content/drive/MyDrive/{candidate}'
    if os.path.exists(p):
        ROOT = p
        break
else:
    ROOT = '/content/drive/MyDrive/eva_lambda'
    os.makedirs(ROOT, exist_ok=True)
print(f'ROOT = {ROOT}')
print('Files:', os.listdir(ROOT))
print('Files on Drive:', os.listdir(ROOT))

In [ ]:
# === 2. Clone repo ===
import os, sys
os.chdir('/content')
if os.path.exists('EVA-Ai'):
    !rm -rf EVA-Ai

!git clone https://github.com/BlackCatSpb/EVA-Ai.git
os.chdir('/content/EVA-Ai')
sys.path.insert(0, '/content/EVA-Ai')
print('CWD:', os.getcwd())

In [ ]:
# === 3. Copy data + checkpoint from Drive ===
import shutil

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('checkpoints_zk', exist_ok=True)

for fname, dest in [
    ('russian_chunks.npy', 'russian_chunks.npy'),
    ('model_step25000.pt', 'checkpoints/model_step25000.pt'),
]:
    src = os.path.join(ROOT, fname)
    if os.path.exists(src):
        sz = os.path.getsize(src) / 1e9
        print(f'Copying {fname} ({sz:.2f} GB)...')
        shutil.copy2(src, dest)

for fn in ['russian_chunks.npy', 'checkpoints/model_step25000.pt']:
    if os.path.exists(fn):
        print(f'  OK: {fn} ({os.path.getsize(fn)/1e9:.2f} GB)')
    else:
        print(f'  MISSING: {fn}')

In [ ]:
# === 4. Check GPU ===
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU not available! Runtime -> Change runtime type -> T4 GPU')

### Тренировка (~2-3 часа)

`train_phase2_zk.py`:
- Загружает чекпоинт `model_step25000.pt` (эмбеддинги + stack)
- Заменяет lm_head на ZeckendorfReadout (свежий)
- Тренирует всё вместе 3 эпохи
- Чекпоинты в `checkpoints_zk/`

In [ ]:
# === 5. Train Phase2-ZK ===
!python train_phase2_zk.py \
    --data russian \
    --train_chunks 10000 \
    --eval_chunks 500 \
    --epochs 3 \
    --resume checkpoints/model_step25000.pt

In [ ]:
# === 6. Compare: Zeckendorf vs lm_head ===
import sys, os, glob, torch, numpy as np, math
sys.path.insert(0, '/content/EVA-Ai')
os.chdir('/content/EVA-Ai')

from ld_model.core import LDConfig, LDStack
from ld_model.readout import ZeckendorfReadout
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda'; D=896; VOCAB=50000
cfg = LDConfig()
cfg.D = D; cfg.n_layers = 12; cfg.n_modes = 4; cfg.vocab = VOCAB
cfg.bottleneck = 256; cfg.adaptive_depth = True; cfg.learnable_V = True; cfg.V_rank = 8

# --- Phase2 (lm_head) ---
print('Loading Phase2 (lm_head)...')
class Phase2Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D)
        self.stack = LDStack(cfg)
        self.lm_head = nn.Linear(D, VOCAB, bias=False)
    def forward(self, x):
        return self.lm_head(self.stack(self.embed(x)))

model_lm = Phase2Model().to(DEVICE)
ckpt = torch.load('checkpoints/model_step25000.pt', map_location=DEVICE, weights_only=True)
sd = ckpt.get('model_fp16', ckpt.get('model_state_dict', ckpt))
sd = {k: v.float() if hasattr(v, 'dtype') and v.dtype == torch.float16 else v for k, v in sd.items()}
model_lm.load_state_dict({k: v for k, v in sd.items() if k in model_lm.state_dict()}, strict=False)
model_lm.eval()
print('  done')

# --- Phase2-ZK (Zeckendorf) ---
print('Loading Phase2-ZK...')
class Phase2ZKModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D)
        self.stack = LDStack(cfg)
        self.readout = ZeckendorfReadout(cfg)
    def forward(self, x):
        return self.stack(self.embed(x))
    def compute_loss(self, x):
        h = self.forward(x)
        log_p = self.readout.log_probs_for_target(h.reshape(-1, D), x.reshape(-1))
        return -log_p.mean()

model_zk = Phase2ZKModel().to(DEVICE)
zk_ckpts = sorted(glob.glob('checkpoints_zk/phase2zk_*.pt'))
if zk_ckpts:
    sd_z = torch.load(zk_ckpts[-1], map_location=DEVICE, weights_only=True)
    sd_z = sd_z.get('model_state_dict', sd_z)
    model_zk.load_state_dict({k: v for k, v in sd_z.items() if k in model_zk.state_dict()}, strict=False)
    print(f'  loaded: {os.path.basename(zk_ckpts[-1])}')
else:
    print('  WARNING: no checkpoint, using fresh init')
model_zk.eval()

# --- Eval ---
arr = np.load('russian_chunks.npy')
loss_lm, loss_zk = 0.0, 0.0
for i in range(50):
    idx = np.random.randint(0, len(arr), 4)
    x = torch.from_numpy(arr[idx, :128].copy()).long().to(DEVICE)
    with torch.no_grad():
        logits = model_lm(x)
        loss_lm += F.cross_entropy(logits.reshape(-1, VOCAB), x.reshape(-1)).item()
        loss_zk += model_zk.compute_loss(x).item()

loss_lm /= 50; loss_zk /= 50
ppl_lm = math.exp(loss_lm); ppl_zk = math.exp(loss_zk)

print(f'\n{"="*55}')
print(f'{"Architecture":<30} {"Params":<12} {"NLL":<10} {"PPL":<10}')
print(f'{"-"*55}')
print(f'{"lm_head":<30} {"44,800,000":<12} {loss_lm:<10.4f} {ppl_lm:<10.1f}')
print(f'{"ZeckendorfReadout":<30} {"86,016":<12} {loss_zk:<10.4f} {ppl_zk:<10.1f}')
print(f'{"Ratio (lm/zk)":<30} {"521x":<12} {"":>10} {ppl_lm/ppl_zk:>10.2f}x')
print(f'{"="*55}')

if ppl_zk < ppl_lm:
    print(f'\n>>> WINNER: ZeckendorfReadout (521x fewer params, {ppl_lm/ppl_zk:.2f}x better PPL)')
else:
    print(f'\n>>> WINNER: lm_head')

In [ ]:
# === 7. Save results to Drive ===
import shutil, glob

for fn in glob.glob('checkpoints_zk/*.pt'):
    dest = os.path.join(ROOT, os.path.basename(fn))
    shutil.copy2(fn, dest)
    print(f'Backed up {os.path.basename(fn)}')

results = f"""Phase2-ZK Comparison Results
{"="*40}
lm_head (44.8M):  NLL={loss_lm:.4f}  PPL={ppl_lm:.1f}
Zeckendorf (86K): NLL={loss_zk:.4f}  PPL={ppl_zk:.1f}
Ratio: {ppl_lm/ppl_zk:.2f}x in favor of {'Zeckendorf' if ppl_zk < ppl_lm else 'lm_head'}
"""
with open(os.path.join(ROOT, 'zk_vs_lm_results.txt'), 'w') as f:
    f.write(results)
print('\nResults saved to Drive')